# Chapter 3: The Coffee Lab — Error Audit

This notebook accompanies **Chapter 3** of the lecture notes.

**Agenda**

🔧 · 📐 · ⚖️ · 📊 · 🏁

**Next steps (take it from here):** 🔄 · 🎯

> **Tip:** Run cells top to bottom. Later cells depend on earlier ones.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from checks import (check_brute_force, check_conditioning,
                    check_stability, check_residuals)


def tufte_axis(ax):
    """Remove spines, keep only outward ticks on left and bottom."""
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(axis='both', which='both', direction='out',
                   length=5, width=1.2, colors='black',
                   top=False, right=False)

## 🔧 How Much Can You Trust Your Coffee Model?

Your coffee lab has a linear quality model: `quality = w * brew_strength + b`. You brewed two coffee samples at different strengths and recorded their quality scores. A brute-force numerical solver will try to recover the model parameters — but how good is the approximation? And if your quality measurements shift slightly, do the parameters fall apart or stay put?

In this notebook you audit a single numerical solution from four angles: the solver itself, the problem's sensitivity to input changes, the algorithm's stability under parameter perturbation, and the residuals left behind.

> A numerical answer always carries some error. The question is never "is there error?" but "how much error, and where does it come from?"

<details><summary>Thought</summary>

Error has at least three sources: discretisation (the grid is too coarse to land on the true answer), conditioning (the problem amplifies input noise), and instability (the algorithm amplifies intermediate rounding). Separating these sources is the whole point of error analysis. In a coffee lab, measurement noise on quality scores is inevitable — the question is whether your model parameters are robust to it.
</details>

## The Coffee Quality System

We work with a 2x2 linear system throughout this notebook. Two coffee samples were brewed at different strengths (X = [2, 5]) and their quality scores were recorded (Y = [11, 13]). The system `[2,1; 5,1] * [w; b] = [11; 13]` lets us solve for the model parameters w (weight on brew strength) and b (baseline quality). The coefficient matrix and right-hand side are defined below, along with the exact solution for reference.

In [ ]:
# Linear system: [[2, 1], [5, 1]] @ [w, b] = [11, 13]
X = np.array([2.0, 5.0])
Y = np.array([11.0, 13.0])

# Exact solution: w = 2/3, b = 29/3
w_exact = 2 / 3
b_exact = 29 / 3

print(f"X = {X}")
print(f"Y = {Y}")
print(f"Exact solution: w = {w_exact:.6f}, b = {b_exact:.6f}")
print()
print("Verification:")
print(f"  2*w + 1*b = {2 * w_exact + 1 * b_exact:.6f}  (should be 11)")
print(f"  5*w + 1*b = {5 * w_exact + 1 * b_exact:.6f}  (should be 13)")

### 🔧 Brute-Force Solver

> The exact solution has w = 2/3 ≈ 0.667. If your grid step h is 1.0, the closest grid point is w = 1.0 — already 50% off. What does this tell you about the relationship between grid resolution and how accurately you can pin down the brew-strength weight?

<details><summary>Thought</summary>

A brute-force grid search can never be more accurate than half the step size in any dimension. With h = 1.0, the best you can hope for is landing within 0.5 of the true value. The error is not random — it is a direct consequence of discretisation. Shrinking h improves accuracy but increases the number of evaluations quadratically. In our coffee lab, a coarse grid means we cannot distinguish between nearby model parameters, so the predicted quality scores will be off.
</details>

Sweep w and b over their ranges with step size h. For each (w, b) pair, compute the predicted quality scores and the mean absolute error against the observed scores Y. Return the best pair and its error.

Useful operations: `np.arange()`, `np.mean()`, `np.abs()`.

In [ ]:
def brute_force_solve(X, Y, w_range, b_range, h):
    """Grid-search for the (w, b) pair with lowest MAE."""
    A = np.array([[X[0], 1], [X[1], 1]])
    best_w, best_b, best_err = None, None, float('inf')
    for w in np.arange(w_range[0], w_range[1] + h, h):
        for b in np.arange(b_range[0], b_range[1] + h, h):
            preds = A @ np.array([w, b])
            err = np.mean(np.abs(Y - preds))
            if err < best_err:
                best_err = err
                best_w = w
                best_b = b
    return (best_w, best_b, best_err)


check_brute_force(brute_force_solve, X, Y, (0, 10), (0, 10), 1.0)

### 📐 Conditioning

> Imagine your first coffee taster revises their quality score by just 0.1 points. You re-solve the system for w and b. If the model parameters barely move, the problem is well-conditioned. If they jump, your coffee quality model amplifies measurement noise. Before running the code — does a system with brew strengths [2, 5] look like it might be sensitive?

<details><summary>Thought</summary>

The two rows [2, 1] and [5, 1] differ only in the brew-strength coefficient. They are not nearly parallel, so the determinant (2·1 − 5·1 = −3) is comfortably away from zero. This suggests moderate conditioning — small perturbations in quality scores will cause proportional, not explosive, changes in the recovered model parameters. A nearly singular system (determinant close to zero) would mean two brew strengths that give almost the same information — a poorly designed experiment.
</details>

For each perturbation value p, add p to Y[0] (the first sample's quality score), solve the perturbed system using np.linalg.solve, and return an array of norm differences between the perturbed and exact solutions.

Useful operations: `np.linalg.solve()`, `np.linalg.norm()`, `array.copy()`.

In [ ]:
perturbations = np.array([-0.5, -0.1, -0.01, 0.01, 0.1, 0.5])


def conditioning(X, Y, perturbations):
    """Return norm of solution change for each perturbation of Y[0]."""
    A = np.array([[X[0], 1], [X[1], 1]])
    exact = np.linalg.solve(A, Y)
    changes = []
    for p in perturbations:
        Y_pert = Y.copy()
        Y_pert[0] += p
        sol_pert = np.linalg.solve(A, Y_pert)
        changes.append(np.linalg.norm(sol_pert - exact))
    return np.array(changes)


check_conditioning(conditioning, X, Y, perturbations)

Let's visualise how the model parameters change as a function of the perturbation in the first coffee sample's quality score.

In [ ]:
cond_result = conditioning(X, Y, perturbations)

if cond_result is not None:
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(perturbations, cond_result, 'ko-', markersize=5, linewidth=1)
    ax.set_xlabel('Perturbation of Y[0]')
    ax.set_ylabel('Norm of solution change')
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5)
    tufte_axis(ax)
    plt.tight_layout()
    plt.show()
else:
    print("Implement conditioning() first.")

**Observe:**
- The relationship between perturbation size and solution change is roughly linear — the coffee quality model is well-conditioned.
- A perturbation of 0.5 in a quality score shifts the model parameters by a moderate amount, not by orders of magnitude.
- For a nearly singular system (e.g., two brew strengths that are almost identical), this curve would be much steeper — your model parameters would be unreliable.

### ⚖️ Stability

> Conditioning asks "does the coffee quality model amplify measurement noise?" Stability asks a different question: "does the algorithm amplify parameter noise?" If you nudge the brew-strength weight w by a tiny amount and the predicted quality scores barely change, the computation is stable. Why is it important to perturb w and b separately rather than together?

<details><summary>Thought</summary>

Perturbing both simultaneously mixes two effects: you cannot tell whether the quality-score change came from w or from b. By isolating each parameter, you learn which one the predictions are more sensitive to. In our coffee model, the coefficient of w (brew strength) ranges from 2 to 5, so a perturbation of w gets amplified more than the same perturbation of b (the baseline, whose coefficient is always 1).
</details>

For each perturbation p, compute predictions with (w+p, b) and with (w, b+p) separately. Compare each to the unperturbed prediction and return the maximum absolute change across both.

Useful operations: `np.max()`, `np.abs()`.

In [ ]:
def stability(X, w, b, perturbations):
    """Return max absolute output change for each perturbation."""
    ref = w * X + b
    results = []
    for p in perturbations:
        pred_w = (w + p) * X + b
        pred_b = w * X + (b + p)
        change_w = np.max(np.abs(pred_w - ref))
        change_b = np.max(np.abs(pred_b - ref))
        results.append(max(change_w, change_b))
    return np.array(results)


check_stability(stability, X, w_exact, b_exact, perturbations)

### 📊 Residuals

> The brute-force solver returned an approximate (w, b) for our coffee model. The residuals tell you how far off the predicted quality scores are from the actual observed scores. If both residuals are zero, the model fits the data perfectly. But with a coarse grid, they won't be zero — their pattern reveals whether the prediction error is systematic or balanced.

<details><summary>Thought</summary>

If both residuals have the same sign, the model consistently over- or under-predicts quality — the grid point is on one side of the true solution. If they have opposite signs, the approximate solution sits between the two constraints, over-predicting one sample and under-predicting the other. The magnitude tells you how far off each quality prediction is.
</details>

Compute the residuals as Y minus the predictions (w*X + b).

Useful operations: basic arithmetic with arrays.

In [ ]:
def residuals(X, Y, w, b):
    """Return the residual vector Y - (w*X + b)."""
    preds = w * X + b
    return Y - preds


# Test with the brute-force solution
bf_result = brute_force_solve(X, Y, (0, 10), (0, 10), 1.0)
if bf_result is not None:
    w_bf, b_bf, _ = bf_result
    check_residuals(residuals, X, Y, w_bf, b_bf)
else:
    check_residuals(residuals, X, Y, 1.0, 10.0)  # fallback test values

Let's visualise the residuals as a bar chart. We compare the brute-force approximation against the exact solution for both coffee samples.

In [ ]:
r_exact = residuals(X, Y, w_exact, b_exact)

if bf_result is not None:
    r_bf = residuals(X, Y, w_bf, b_bf)
else:
    r_bf = None

if r_exact is not None:
    fig, ax = plt.subplots(figsize=(6, 4))
    bar_x = np.arange(len(X))
    width = 0.35

    ax.bar(bar_x - width / 2, r_exact, width, label='Exact solution',
           color='tab:blue', edgecolor='none')
    if r_bf is not None:
        ax.bar(bar_x + width / 2, r_bf, width, label='Brute-force (h=1.0)',
               color='tab:orange', edgecolor='none')

    ax.set_xlabel('Equation index')
    ax.set_ylabel('Residual')
    ax.set_xticks(bar_x)
    ax.set_xticklabels(['Eq. 1 (2w + b = 11)', 'Eq. 2 (5w + b = 13)'])
    ax.axhline(0, color='black', linewidth=0.5)
    ax.legend(frameon=False, fontsize=9)
    tufte_axis(ax)
    plt.tight_layout()
    plt.show()
else:
    print("Implement residuals() first.")

**Observe:**
- The exact solution produces residuals of zero (within floating-point precision) — the model fits both coffee samples perfectly, as expected.
- The brute-force solution shows non-zero residuals whose size depends on how far the grid point is from the true model parameters.
- This is discretisation error, not a bug. Shrinking h would reduce these residuals and improve the quality predictions.

### 🏁 Recap

**What we did:**
- 🔧 Built a brute-force grid solver for our coffee quality model and saw how step size limits accuracy.
- 📐 Probed conditioning by perturbing a quality score and measuring how much the model parameters moved.
- ⚖️ Tested stability by perturbing the parameters and tracking how predicted quality scores changed.
- 📊 Computed residuals to quantify how far off an approximate model's predictions really are.

**Key takeaways:**
- Every numerical answer carries error. The job is to measure it, not ignore it.
- Conditioning is a property of the problem (how your coffee experiment was designed); stability is a property of the algorithm. Both matter.
- Residuals are the simplest sanity check — if the predicted quality scores are far from the observed ones, something is wrong, regardless of the source.

**Now head back for self-check questions and key learnings in the lecture notes.**

## Take It from Here — Next Steps (Optional)

The exercises below are **optional** extensions. They deepen your intuition but are not required to follow the rest of the course. Work through them at your own pace after the session.

### 🔄 Convergence — Refining h

> The brute-force solver with h = 1.0 gave a rough approximation of the coffee model parameters. What happens as you halve h repeatedly? Does the error decrease predictably, and at what computational cost?

<details><summary>Thought</summary>

Each time you halve h, the number of grid points doubles in each dimension, so the total work quadruples. But the maximum possible error also halves (the grid gets finer). This is a classic accuracy-vs-cost tradeoff. Plotting the error as a function of h should show a roughly linear decrease on a log-log scale. In the coffee lab, finer grids let you pin down w and b more precisely — but the tasting budget is finite.
</details>

Run the brute-force solver for several values of h and plot the error.

In [ ]:
h_values = [2.0, 1.0, 0.5, 0.25, 0.1]
errors = []

for h in h_values:
    result = brute_force_solve(X, Y, (0, 10), (0, 10), h)
    if result is not None:
        _, _, err = result
        errors.append(err)
    else:
        errors.append(None)

if all(e is not None for e in errors):
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(h_values, errors, 'ko-', markersize=5, linewidth=1)
    ax.set_xlabel('Step size h')
    ax.set_ylabel('Best MAE')
    tufte_axis(ax)
    plt.tight_layout()
    plt.show()
else:
    print("Implement brute_force_solve() first.")

### 🎯 Condition Number

> The condition number of the coefficient matrix summarises the sensitivity of your coffee model in a single number. A condition number of 1 is perfect; large values signal danger. Compute the condition number and relate it to the conditioning plot above.

<details><summary>Thought</summary>

np.linalg.cond returns the ratio of the largest to smallest singular value. For our matrix [[2, 1], [5, 1]] (built from the two brew strengths), this number is moderate (around 4-5), confirming that the model is well-conditioned. A condition number of 1000 would mean that a perturbation of 0.01 in a quality score could shift the model parameters by up to 10 — that is a coffee experiment you should redesign with more distinct brew strengths.
</details>

Useful operation: `np.linalg.cond()`.

In [ ]:
A = np.array([[X[0], 1], [X[1], 1]])
kappa = np.linalg.cond(A)
print(f"Coefficient matrix:\n{A}")
print(f"\nCondition number: {kappa:.4f}")
print(f"\nInterpretation: a perturbation of size p in the input can cause")
print(f"a solution change of up to {kappa:.1f} * p in the worst case.")